In [ ]:
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pertpy as pt

In [ ]:
adata = sc.read("../results/adata/09-annotation.h5ad")

In [ ]:
# Count cells per sample and cluster
counts = (
    adata.obs
    .groupby(["sample", "leiden_0.15"])
    .size()
    .reset_index(name="n_cells")
)

# Total cells per sample
totals = (
    adata.obs
    .groupby("sample")
    .size()
    .reset_index(name="total_cells")
)

# Merge and compute proportions
df = counts.merge(totals, on="sample")
df["proportion"] = df["n_cells"] / df["total_cells"]

# Add day information (one value per sample)
sample_info = (
    adata.obs[["sample", "day"]]
    .drop_duplicates()
)

df = df.merge(sample_info, on="sample")

In [ ]:
g = sns.catplot(
    data=df,
    x="day",
    y="proportion",
    col="leiden_0.15",
    # kind="box",
    hue="day",
    col_wrap=4,
    sharey=False,
    height=3
)
plt.show()

In [ ]:

sns.barplot(data=df, x="leiden_0.15", y="proportion", hue="day")
plt.show()

In [ ]:
sccoda_model = pt.tl.Sccoda()
sccoda_data = sccoda_model.load(
    adata,
    type="cell_level",
    generate_sample_level=True,
    cell_type_identifier="leiden_0.15",
    sample_identifier="sample",
    covariate_obs=["day"]
)


In [ ]:
sccoda_model.plot_boxplots(sccoda_data, feature_name="day", add_dots=True)
plt.show()

In [ ]:
# sccoda_model.plot_stacked_barplot(adata, feature_name="day")
# plt.show()

In [ ]:
sccoda_data = sccoda_model.prepare(sccoda_data, formula="day")

In [ ]:
sccoda_model.run_nuts(sccoda_data)

In [ ]:
sccoda_model.summary(sccoda_data)

In [ ]:
sccoda_model.credible_effects(sccoda_data)

In [ ]:
sccoda_model.plot_effects_barplot(sccoda_data, parameter="Final Parameter")

In [ ]:
sccoda_model.set_fdr(sccoda_data, est_fdr=0.2)
sccoda_model.summary(sccoda_data)

In [ ]:
sccoda_model.credible_effects(sccoda_data)

In [ ]:
sccoda_model.plot_effects_barplot(sccoda_data, parameter="Final Parameter")